## 将格式转换为next_token需要的格式 张: 人物名称, 三: 人物名称 nt后缀文件

In [ ]:
import json
import os

prompt = '指令: 请识别输入句子中属于实体类别列表的命名实体, 实体类别列表: gpe.、gpe.、loc.、loc.、org.、org.、per.、per., 未被识别为上述八类的字符，统一标记为"not." 输出格式要求: 1. 按照输入序列的顺序, 一个字符对应一个标签, 例如："张: per., 三: per." 2. 输出的标签必须包含于九个候选标签中 3. 每个字符必须被标注, 不允许跳过 输入: '
def convert_entities(text_data):
    original_text = ''.join(text_data['text'])
    # 初始化所有字符为"不是实体"
    labels = ["not."] * len(text_data["text"])

    # 遍历所有实体进行标记
    for entity in text_data["entities"]:
        start = entity["start"]
        end = entity["end"]
        entity_type = entity["entity"]
        
        # 转换为中文标签
        
        chinese_label = {
            "GPE.NAM": "gpe.",
            "GPE.NOM": "gpe.",
            "LOC.NAM": "loc.",
            "LOC.NOM": "loc.",
            "ORG.NAM": "org.",
            "ORG.NOM": "org.",
            "PER.NAM": "per.",
            "PER.NOM": "per."
        }.get(entity_type, "not.")
        
        # 标记实体范围内的字符
        for i in range(start, end):
            if i < len(labels):
                labels[i] = chinese_label

    # 生成标注字符串
    annotations = []
    for char, label in zip(text_data["text"], labels):
        annotations.append(f"{char}: {label}")
    
    # 构建目标格式
    return {
        "conversations": [
            {
                "role": "user",
                "content": prompt + original_text
            },
            {
                "role": "assistant",
                "content": ", ".join(annotations)
            }
        ]
    }

def process_jsonl(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile, \
        open(output_path, 'w', encoding='utf-8') as outfile:
        for line in infile:
            # 解析每行JSON对象
            data = json.loads(line.strip())
                
            # 这里可以添加数据处理逻辑（例如您之前的实体转换）
            processed_data = convert_entities(data)
                
            # 写入处理后的数据
            outfile.write(json.dumps(processed_data, ensure_ascii=False) + '\n')

process_jsonl('/home/datasets/few_shot/weibo/train_1000.jsonl', './datasets/weibo/train_1000_nt.jsonl')


## 张(人物名称)三(人物名称) nt2后缀文件

In [4]:
import json
import os

prompt = '指令: 请识别输入句子中属于实体类别列表的命名实体, 实体类别列表: 所属国籍、教育背景、籍贯地域、个人姓名、组织机构、专业领域、民族类别、职称名称, 未被识别为上述八类的字符，统一标记为"不是实体" 输出格式要求: 1. 按照输入序列的顺序, 一个字符对应一个标签, 例如："张(个人姓名)三(个人姓名)" 2. 输出的标签必须包含于九个候选标签中 3. 每个字符必须被标注, 不允许跳过 4. 非实体统一用"不是实体", 不使用其他表述 输入: '
def convert_entities(text_data):
    original_text = ''.join(text_data['text'])
    # 初始化所有字符为"不是实体"
    labels = ["不是实体"] * len(text_data["text"])

    # 遍历所有实体进行标记
    for entity in text_data["entities"]:
        start = entity["start"]
        end = entity["end"]
        entity_type = entity["entity"]
        
        # 转换为中文标签
        
        chinese_label = {
            "CONT": "所属国籍",
            "EDU": "教育背景",
            "LOC": "籍贯地域",
            "NAME": "个人姓名",
            "ORG": "组织机构",
            "PRO": "专业领域",
            "RACE": "民族类别",
            "TITLE": "职称名称"
        }.get(entity_type, "不是实体")
        
        # 标记实体范围内的字符
        for i in range(start, end):
            if i < len(labels):
                labels[i] = chinese_label

    # 生成标注字符串
    annotations = []
    for char, label in zip(text_data["text"], labels):
        annotations.append(f"{char}({label})")
    
    # 构建目标格式
    return {
        "conversations": [
            {
                "role": "user",
                "content": prompt + original_text
            },
            {
                "role": "assistant",
                "content": "".join(annotations)
            }
        ]
    }

def process_jsonl(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile, \
        open(output_path, 'w', encoding='utf-8') as outfile:
        for line in infile:
            # 解析每行JSON对象
            data = json.loads(line.strip())
                
            # 这里可以添加数据处理逻辑（例如您之前的实体转换）
            processed_data = convert_entities(data)
                
            # 写入处理后的数据
            outfile.write(json.dumps(processed_data, ensure_ascii=False) + '\n')

process_jsonl('/home/datasets/few_shot/resume/train_1350.jsonl', './datasets/resume/train_1350_nt2.jsonl')


## 将格式转换为原始的entity type格式

In [7]:
import json
import os

prompt = '指令: 请识别输入句子中属于实体类别列表的命名实体, 实体类别列表: 所属国籍、教育背景、籍贯地域、个人姓名、组织机构、专业领域、民族类别、职称名称 输出格式要求: 1. 输出格式为[{"entity": "", "type": ""}], 其中"entity"表示所提取的实体文本, "type"表示所提取的实体类型, 一个entity对应一个type 2. 输出的标签必须包含于八个候选标签中 3. 如果不存在任何实体，请输出空数组[] 输入: '
# 映射表
label_map = {
    "CONT": "所属国籍",
    "EDU": "教育背景",
    "LOC": "籍贯地域",
    "NAME": "个人姓名",
    "ORG": "组织机构",
    "PRO": "专业领域",
    "RACE": "民族类别",
    "TITLE": "职称名称"
}
def convert_entities(text_data):
    text_chars = text_data['text']
    original_text = ''.join(text_chars)

    new_entities = []
    # 遍历所有实体进行标记
    for entity in text_data["entities"]:
        start = entity["start"]
        end = entity["end"]
        ent_text = ''.join(text_chars[start:end])
        ent_type = label_map.get(entity['entity'], entity['entity'])
        new_entities.append({
            "entity": ent_text,
            "type": ent_type
        })
    
    # 构建目标格式
    return {
        "conversations": [
            {
                "role": "user",
                "content": prompt + original_text
            },
            {
                "role": "assistant",
                "content": json.dumps(new_entities, ensure_ascii=False)
            }
        ]
    }

def process_jsonl(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile, \
        open(output_path, 'w', encoding='utf-8') as outfile:
        for line in infile:
            # 解析每行JSON对象
            data = json.loads(line.strip())
                
            # 这里可以添加数据处理逻辑（例如您之前的实体转换）
            processed_data = convert_entities(data)
                
            # 写入处理后的数据
            outfile.write(json.dumps(processed_data, ensure_ascii=False) + '\n')

process_jsonl('/home/datasets/few_shot/resume/dev.jsonl', './datasets/resume/dev_lora.jsonl')


## nt2格式添加gold_answers和bad_answers，gold_answers为assistant content,bad_answers为空

In [3]:
import json

def add_preferences_to_jsonl(input_path: str, output_path: str) -> None:
    """
    读取 .jsonl 文件，为每一行添加 "gold_answers" 和 "bad_answers" 两个键，
    并将结果写入新的 .jsonl 文件。

    - gold_answers：值为原来 assistant 的 content
    - bad_answers ：值为 ""（空字符串）
    """
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            # 解析 JSON
            data = json.loads(line)

            # 从 conversations 中找到第一个 assistant 内容
            gold = ""
            for turn in data.get("conversations", []):
                if turn.get("role") == "assistant":
                    gold = turn.get("content", "")
                    break

            # 添加键
            data["gold_answers"] = gold
            data["bad_answers"] = ""

            # 写回 .jsonl（每行一个 JSON 对象）
            fout.write(json.dumps(data, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    # 示例用法
    input_file  = "./datasets/ud/test_nt2.jsonl"
    output_file = "./datasets/ud/test_dpo_nobad.jsonl"
    add_preferences_to_jsonl(input_file, output_file)


## 计算采样结果的BLEU，选择bleu<1且最小的作为bad_answers，可能每个结果都=1，那么bad_answers还是空

In [1]:
import os
import json
from typing import List
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

def load_txt_lines(file_path: str) -> List[str]:
    """读取一个 .txt 文件，返回所有非空行的列表。"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

def compute_max_bleu_below_one(candidates: List[str], reference: str) -> (float, str):
    """
    对一组候选字符串计算它们与 reference 的 BLEU‑4 分数，
    返回 (最小 BLEU<1.0, 对应的候选文本)。若所有 BLEU==1.0，返回 (1.0, "")。
    """
    best_score = 2.0
    best_text = ""
    for cand in candidates:
        score = sentence_bleu(
                    [reference],
                    cand,
                    smoothing_function=SmoothingFunction().method3
                )
        # 只考虑 <1.0 的情况
        if score < 1.0 and score < best_score:
            best_score = score
            best_text = cand
    if best_score > 1.0:
        # 说明所有分数都是 1.0
        return 1.0, ""
    return best_score, best_text

def main():
    # 1. 加载 train_1000_dpo_nobad.jsonl
    input_jsonl = "./datasets/ud/dev_dpo_nobad.jsonl"
    with open(input_jsonl, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()]

    # 2. 找到那 20 个 .txt 文件（假设放在同一目录下，以 .txt 结尾，且排除其它 txt）
    data_dir = os.path.dirname("./data_record/predict_ud_1000_nt2_1_46500_dev_sample1/output_nt2_sample0.txt")
    txt_files = [os.path.join(data_dir, fn)
                 for fn in os.listdir(data_dir)
                 if fn.endswith(".txt")]

    # 3. 对每一行记录，依次加载 20 个文件中同一行的所有候选
    num_lines = len(records)
    # 预先将所有 20 个文件的内容一次性读入（每个文件是 list of lines）
    all_txt_lines = [load_txt_lines(path) for path in txt_files]
    # 检查行数一致
    for lines in all_txt_lines:
        assert len(lines) == num_lines, "所有 txt 文件行数必须与 jsonl 相同"

    # 4. 逐行处理
    output_path = "./datasets/ud/dev_dpo.jsonl"
    with open(output_path, 'w', encoding='utf-8') as fout:
        for i, record in enumerate(records):
            gold = record.get("gold_answers", "").strip()
            # 如果 gold 为空，则跳过
            if not gold:
                record["bad_answers"] = ""
            else:
                # 取第 i 行的所有候选
                candidates = [lines[i] for lines in all_txt_lines]
                # 计算最佳 bad answer
                score , bad = compute_max_bleu_below_one(candidates, gold)
                record["bad_answers"] = bad

            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

if __name__ == "__main__":
    main()


## good和bad之间的bleu差值

In [2]:
import json
import re
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu


def calculate_bleu_scores(input_path, output_path):
    """计算并保存BLEU分数"""
    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w', encoding='utf-8') as outfile:
        
        
        for line in infile:
            data = json.loads(line.strip())
            gold = data['gold_answers']
            bad = data['bad_answers']


            score = sentence_bleu(
                    [gold],
                    bad,
                    smoothing_function=SmoothingFunction().method3
            )
            
            # 保存结果（保留原数据并新增bleu_score字段）
            data['bleu_score'] = float(1.0-score)  # 转换为float类型保证可序列化
            outfile.write(json.dumps(data, ensure_ascii=False) + '\n')

# 使用示例
calculate_bleu_scores('./datasets/ud/dev_dpo.jsonl', './datasets/ud/dev_bleu.jsonl')

## 平常格式的lora训练

In [3]:
from main.trainer.llm_lora import Trainer
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained("/home/lpc/models/Llama-3.1-8B-Instruct", trust_remote_code=True)
print([tokenizer.encode(char, add_special_tokens=False)
            for char in ['人','物','名','称']])
print([tokenizer.encode(char, add_special_tokens=False)
            for char in ['其','它','杂','项']])
print([tokenizer.encode(char, add_special_tokens=False)
            for char in ['电','视','节','目']])
print([tokenizer.encode(char, add_special_tokens=False)
            for char in ['不','是','实','体']])
config = AutoConfig.from_pretrained("/home/lpc/models/Llama-3.1-8B-Instruct", trust_remote_code=True)
trainer = Trainer(tokenizer=tokenizer, config=config, from_pretrained='/home/lpc/models/Llama-3.1-8B-Instruct', loader_name='LLM_Chat', data_path='taobao_1000_lora', max_length=3600, batch_size=2, batch_size_eval = 2, task_name='taobao_1000_lora_new_1')

for i in trainer(num_epochs=100, lr=1e-5):
    a = i

[[17792], [53953], [13372], [25666]]
[[42246], [103282], [114223], [48982]]
[[39312], [58552], [56602], [30832]]
[[16937], [21043], [41073], [33014]]


TypeError: string indices must be integers

## next token的lora训练

In [1]:
from main.trainer.llm_lora import Trainer
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained("/home/glm-4-9b-chat", trust_remote_code=True)
config = AutoConfig.from_pretrained("/home/glm-4-9b-chat", trust_remote_code=True)
trainer = Trainer(tokenizer=tokenizer, config=config, from_pretrained='/home/glm-4-9b-chat', loader_name='LLM_Chat', data_path='weibo_1000_nt2', max_length=3600, batch_size=2, batch_size_eval = 2, task_name='weibo_1000_nt2_new_1')

for i in trainer(num_epochs=100, lr=1e-5):
    a = i

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AutoModel Choose Model: /home/glm-4-9b-chat



Loading checkpoint shards: 100%|██████████| 10/10 [00:05<00:00,  1.68it/s]


trainable params: 5,570,560 || all params: 9,405,521,920 || trainable%: 0.0592


  0%|          | 0/500 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 370.00 MiB (GPU 0; 44.42 GiB total capacity; 42.77 GiB already allocated; 282.88 MiB free; 43.24 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

## DPO训练

In [1]:
from main.trainer.llm_dpo import Trainer
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained("/home/glm-4-9b-chat", trust_remote_code=True)
config = AutoConfig.from_pretrained("/home/glm-4-9b-chat", trust_remote_code=True)
trainer = Trainer(tokenizer=tokenizer, config=config, resume_path='./save_model/youku_1000_nt2_new_1/ChatGLM_26500', from_pretrained='/home/glm-4-9b-chat', loader_name='LLM_DPO', data_path='youku_1000_dpo', max_length=3600, batch_size=1, batch_size_eval = 1, task_name='youku_1000_dpo_new_3')
ids1 = [151331, 151333, 151337, 198]
ids1.extend(tokenizer.encode("甘佳欣大美女", add_special_tokens=False))
print(ids1)
print(tokenizer.decode(ids1))
for i in trainer(num_epochs=100, lr=1e-5):
    a = i

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AutoModel Choose Model: /home/glm-4-9b-chat



Loading checkpoint shards: 100%|██████████| 10/10 [00:04<00:00,  2.11it/s]


Accessing Resume PATH: ./save_model/youku_1000_nt2_new_1/ChatGLM_26500 ...

trainable params: 5,570,560 || all params: 9,405,521,920 || trainable%: 0.0592
trainable params: 0 || all params: 9,405,521,920 || trainable%: 0.0000
[151331, 151333, 151337, 198, 100502, 99721, 100975, 98324, 104720]
[gMASK] <sop> <|assistant|> 
甘佳欣大美女


KeyboardInterrupt: 

## 平常格式的推理

In [1]:
from main.predictor.llm_lora import Predictor

pred = Predictor(model_from_pretrained='/home/glm-4-9b-chat', resume_path='./save_model/taobao_1000_lora_new_1/ChatGLM_43000', data_path="taobao_1000_lora")
result = pred(max_length=512, batch_size=20)

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 10/10 [00:04<00:00,  2.14it/s]
/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


=== NER Evaluation ===
Precision: 0.6756
Recall:    0.7367
F1-score:  0.7048


## next token的推理

In [1]:
from main.predictor.llm_nt import Predictor

pred = Predictor(model_from_pretrained='/home/glm-4-9b-chat', resume_path='./save_model/youku_1000_nt_new_1/ChatGLM_17500', data_path="youku_1000_nt")
result = pred(query="指令: 请识别输入句子中属于实体类别列表的命名实体, 实体类别列表: 杂项、人物名称、电视节目, 未被识别为上述三类的字符，统一标记为\"不是实体\" 输出格式要求: 1. 按照输入序列的顺序, 一个字符对应一个标签, 例如：\"张: 人物名称, 三: 人物名称\" 2. 输出的标签必须包含于四个候选标签中 3. 每个字符必须被标注, 不允许跳过 4. 非实体统一用\"不是实体\", 不使用其他表述 输入: 【小宇】我的世界服务器多人小游戏和小云小杰随便玩玩系列=,=", max_length=512, batch_size=2)

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 10/10 [00:04<00:00,  2.12it/s]


['【: 不是实体, 小: 人物名称, 宇: 人物名称, 】: 不是实体, 我: 不是实体, 的: 不是实体, 世: 不是实体, 界: 不是实体, 服: 不是实体, 务: 不是实体, 器: 不是实体, 多: 不是实体, 人: 不是实体, 小: 不是实体, 游: 不是实体, 戏: 不是实体, 和: 不是实体, 小: 不是实体, 云: 不是实体, 小: 不是实体, 杰: 不是实体, 随: 不是实体, 便: 不是实体, 玩: 不是实体, 玩: 不是实体, 系: 不是实体, 列: 不是实体, =: 不是实体, ,: 不是实体, =: 不是实体']


## llm_nt推理文件对应格式是张: 人物名称, 三: 人物名称

In [2]:
from main.predictor.llm_nt import Predictor

pred = Predictor(model_from_pretrained='/home/glm-4-9b-chat', resume_path='./save_model/youku_1000_nt_new_2/ChatGLM_25500', data_path="youku_1000_nt")
result = pred(max_length=512, batch_size=20)

Loading checkpoint shards: 100%|██████████| 10/10 [00:05<00:00,  1.73it/s]


              precision    recall  f1-score   support

        人物名称    0.77417   0.84817   0.80948      1067
        其它杂项    0.28153   0.34968   0.31193       632
        电视节目    0.86730   0.72619   0.79050      3528

   micro avg    0.75143   0.70557   0.72778      5227
   macro avg    0.64100   0.64135   0.63730      5227
weighted avg    0.77746   0.70557   0.73651      5227



## llm_nt2推理文件对应格式是张(人物名称)三(人物名称)

In [2]:
from main.predictor.llm_nt2 import Predictor

pred = Predictor(model_from_pretrained='/home/glm-4-9b-chat', resume_path='./save_model/weibo_1000_nt2_new_2/ChatGLM_11000', data_path="weibo_1000_nt2")
result = pred(max_length=512, batch_size=20)

Loading checkpoint shards: 100%|██████████| 10/10 [00:05<00:00,  1.94it/s]


ValueError: max() arg is an empty sequence

In [1]:
from main.predictor.llm_nt2_en import Predictor

pred = Predictor(model_from_pretrained='/home/glm-4-9b-chat', resume_path='./save_model/conll2003_1000_nt2_new_1/ChatGLM_20000', data_path="conll2003_1000_nt2")
result = pred(max_length=512, batch_size=20)

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 10/10 [00:04<00:00,  2.00it/s]


KeyboardInterrupt: 

## 偏好对采样

In [1]:
from main.predictor.llm_nt2_sample import Predictor

pred = Predictor(model_from_pretrained='/home/glm-4-9b-chat', resume_path='./save_model/youku_1000_nt2_new_2/ChatGLM_38000', data_path="youku_1000_nt2")
result = pred(max_length=512, batch_size=20)

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 10/10 [00:05<00:00,  1.98it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB (GPU 0; 44.42 GiB total capacity; 2.34 GiB already allocated; 15.56 MiB free; 2.34 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

## 将数据集格式转换为PPO格式{"conversations": [{"role": "user", "content": "你的主人是谁？"}, {"role": "assistant", "content": "张三是我的主人。"}], "gold_answers": ["张三是我的主人。"], "bad_answers": ["我没有主人", "我不知道", "我没有真正的主人", "我是人工智能没有主人"]}

In [4]:
import json
import jsonlines

prompt = '指令: 请识别输入句子中属于实体类别列表的命名实体, 并使用JSON格式的数组进行返回, 子项包括entity和type属性 实体类别列表: "misc"、"per"、"television", 其中"misc"表示杂项, "per"表示人物名称, "television"表示节目 格式要求: 1. 输出格式为[{"type": "", "entity": ""}], 其中键"type"表示实体类型, 对应的值只能是实体类别列表中的其中一个, 键"entity"表示实体文本, 对应的值是从输入句子中所提取的实体文本, 键非空时，值也非空, 一个entity对应一个type 2. 如果不存在任何实体, 请直接输出空数组[] 3. 请不要输出额外的内容, 例如转义字符\n等等 输入: '

def convert_jsonl(input_path, output_path):
    with jsonlines.open(input_path, mode='r') as reader:
        with jsonlines.open(output_path, mode='w') as writer:
            for obj in reader:
                # 合并原始文本
                original_text = ''.join(obj['text'])
                
                # 提取实体信息
                entities = []
                for entity in obj['entities']:
                    entity_text = ''.join(entity['text'])
                    entities.append({
                        "type": entity['entity'].lower(),
                        "entity": entity_text
                    })
                
                # 构建对话结构
                converted = {
                    "conversations": [
                        {
                            "role": "user",
                            "content": prompt + original_text
                        },
                        {
                            "role": "assistant",
                            # json的正确格式得用双引号，而字典单引号和双引号都行
                            "content": json.dumps(entities, ensure_ascii=False)  # 转换为字符串，False表示将非ascii字符原样输出
                        }
                    ],
                    "gold_answers": [
                        json.dumps(entities, ensure_ascii=False)
                    ],
                    "bad_answers": ["[]"]
                }
                writer.write(converted)

# 使用示例
convert_jsonl('/home/datasets/few_shot/youku/dev.jsonl', './data/youku/dev_ppo_nobad.jsonl')

In [2]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from main.trainer.chatglm_rlhf import Trainer
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained("/home/chatglm3-6b/", trust_remote_code=True)
config = AutoConfig.from_pretrained("/home/chatglm3-6b/", trust_remote_code=True)
trainer = Trainer(tokenizer=tokenizer, config=config, from_pretrained='/home/chatglm3-6b/', reward_from_pretrained='/home/text2vec-base-chinese/', loader_name='ChatGLM_RLHF', data_path='youku_1000_ppo_nobad', max_length=1200, batch_size=2, task_name='youku_1000_ppo_nobad')

for i in trainer(num_epochs=60, lr=1e-5):
    a = i

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


TypeError: string indices must be integers

In [ ]:
import json
import jsonlines

prompt = '''指令: 请识别输入句子中属于实体类别列表的命名实体, 并使用JSON格式的数组进行返回, 子项包括entity和type属性; 实体类别列表: misc、per、television, 其中misc表示实体类型为杂项, per表示实体类型为人物名称, television表示实体类型为节目
格式要求: 1. 输出格式为[{'type': '', 'entity': ''}], 其中键'type'表示实体类型, 对应的值只能是实体类别列表中的其中一个, 键'entity'表示实体文本, 对应的值是从输入句子中所提取的实体文本, 一个entity对应一个type
2. 如果不存在任何实体, 请直接输出空数组[]
3. 请不要输出额外的内容, 例如转义字符等等
输入: '''

def convert_jsonl(input_path, output_path):
    with jsonlines.open(input_path, mode='r') as reader:
        with jsonlines.open(output_path, mode='w') as writer:
            for obj in reader:
                original_text = ''.join(obj['text'])
                entities = []
                # 生成组合偏移错误实体
                error_entities = []

                for entity in obj['entities']:
                    # 提取实体基本信息
                    entity_text = ''.join(entity['text'])
                    start = entity['start']
                    end = entity['end']
                    entity_type = entity['entity'].lower()
                    
                    # 生成正确实体
                    entities.append({"type": entity_type, "entity": entity_text})
                    
                    # 组合偏移逻辑（网页1][网页7）
                    new_start = max(0, start - 1)  # 左移边界控制
                    new_end = min(len(original_text), end + 1)  # 右移边界控制
                    
                    # 生成包含左右偏移的实体（网页8）
                    combined_entity = original_text[new_start:new_end]
                    if combined_entity != entity_text:  # 排除无效偏移
                        error_entities.append({
                            "type": entity_type,
                            "entity": combined_entity
                        })
                    

                # 构建最终数据结构
                converted = {
                    "conversations": [
                        {"role": "user", "content": prompt + original_text},
                        {"role": "assistant", "content": json.dumps(entities, ensure_ascii=False)}
                    ],
                    "gold_answers": [json.dumps(entities, ensure_ascii=False)],
                    "bad_answers": [json.dumps(error_entities, ensure_ascii=False)]
                }
                writer.write(converted)

convert_jsonl('/home/datasets/few_shot/youku/train_1000.jsonl', './data/youku/train_1000_ppo.jsonl')

In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from main.trainer.chatglm_rlhf import Trainer
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained("/home/chatglm3-6b/", trust_remote_code=True)
config = AutoConfig.from_pretrained("/home/chatglm3-6b/", trust_remote_code=True)
trainer = Trainer(tokenizer=tokenizer, config=config, from_pretrained='/home/chatglm3-6b/', reward_from_pretrained='/home/text2vec-base-chinese/', loader_name='ChatGLM_RLHF', data_path='youku_1000_ppo_nobad', max_length=1200, batch_size=2, task_name='youku_1000_ppo_nobad_6')

for i in trainer(num_epochs=60 ,lr=1e-5):
    a = i

/root/miniconda3/envs/chatglmm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


AutoModel Choose Model: /home/chatglm3-6b/



Loading checkpoint shards: 100%|██████████| 7/7 [00:01<00:00,  4.01it/s]
Some weights of the model checkpoint at /home/chatglm3-6b/ were not used when initializing ChatGLMForConditionalGeneration: ['transformer.encoder.layers.16.post_attention_layernorm.weight', 'transformer.encoder.layers.7.mlp.dense_h_to_4h.weight', 'transformer.encoder.layers.23.self_attention.dense.weight', 'transformer.encoder.layers.11.mlp.dense_h_to_4h.weight', 'transformer.encoder.layers.16.self_attention.dense.weight', 'transformer.encoder.layers.18.mlp.dense_4h_to_h.weight', 'transformer.encoder.layers.4.mlp.dense_4h_to_h.weight', 'transformer.encoder.layers.26.input_layernorm.weight', 'transformer.encoder.layers.12.self_attention.dense.weight', 'transformer.encoder.layers.23.input_layernorm.weight', 'transformer.encoder.layers.8.post_attention_layernorm.weight', 'transformer.encoder.layers.19.mlp.dense_h_to_4h.weight', 'transformer.encoder.layers.16.input_layernorm.weight', 'transformer.encoder.layers.5.mlp.

trainable params: 3,899,392 || all params: 6,247,483,392 || trainable%: 0.0624


  0%|          | 0/500 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 1.621 seconds.
Prefix dict has been built successfully.
Train: 7/60:  30%|███       | 151/500 [05:45<13:19,  2.29s/it, bleu-4=0.473, rouge-1=51.8, rouge-2=45.6, rouge-l=55.1, train_loss=1.67e+11]


KeyboardInterrupt: 